# Portfolio Optimization Exploration

This notebook explores mean-variance portfolio optimization using the sma-quant-core framework.

**Notebook purpose**: Research and experimentation
- Load sample data
- Experiment with optimization objectives
- Generate efficient frontier
- Analyze sensitivity to parameters
- Document findings for decision memo

In [ ]:
import json
import numpy as np
import pandas as pd
from datetime import datetime

from src.mean_variance_optimizer import MeanVarianceOptimizer, EfficientFrontier
from sma_quant_core.models import Asset, PortfolioConstraints, PortfolioConstraint

print("Imports successful")

## 1. Load Sample Data

In [ ]:
# Load assets
with open("data/sample_assets.json") as f:
    assets_data = json.load(f)

assets = [Asset(**a) for a in assets_data]

# Display assets
print("Assets")
print("="*80)
for asset in assets:
    print(f"{asset.id:10} | {asset.name:45} | Return: {asset.expected_return:.1%}  Vol: {asset.volatility:.1%}")

print(f"\nTotal assets: {len(assets)}")

In [ ]:
# Load constraints
with open("data/sample_constraints.json") as f:
    constraints_data = json.load(f)

constraints = PortfolioConstraints(budget_constraint=constraints_data.get("budget_constraint", 1.0))
for c in constraints_data.get("constraints", []):
    constraints.constraints.append(PortfolioConstraint(**c))

print("Constraints")
print("="*80)
for i, c in enumerate(constraints.constraints, 1):
    print(f"{i}. {c.constraint_type:20} | Asset: {c.asset_id or 'Portfolio':10} | Lower: {c.lower_bound} | Upper: {c.upper_bound}")

## 2. Basic Optimization: Maximize Sharpe Ratio

In [ ]:
# Create optimizer (fallback mode for reproducibility without cvxpy)
opt = MeanVarianceOptimizer(use_cvxpy=False)

# Optimize for max Sharpe ratio
portfolio = opt.optimize(
    assets,
    constraints,
    objective="max_sharpe",
    risk_free_rate=0.02
)

print("Optimized Portfolio (Max Sharpe Ratio)")
print("="*80)
print(f"Expected Return:   {portfolio.expected_return:.2%}")
print(f"Expected Volatility: {portfolio.expected_volatility:.2%}")
print(f"Sharpe Ratio:      {portfolio.sharpe_ratio:.4f}")
print(f"Status:            {portfolio.status}")

if portfolio.warnings:
    print(f"\nWarnings:")
    for warning in portfolio.warnings:
        print(f"  - {warning}")

print(f"\nAllocations:")
for asset_id, weight in sorted(portfolio.weights.items(), key=lambda x: -x[1]):
    print(f"  {asset_id:10}: {weight:6.2%}")

## 3. Efficient Frontier Analysis

In [ ]:
# Generate efficient frontier
frontier_gen = EfficientFrontier(opt)
frontier_points = frontier_gen.generate(
    assets,
    constraints,
    n_points=15
)

print("Efficient Frontier")
print("="*80)
print(f"{'Return':<10} {'Volatility':<15} {'Sharpe Ratio':<15}")
print("-"*80)

for point in frontier_points:
    print(f"{point['return']:<10.2%} {point['volatility']:<15.2%} {point['sharpe']:<15.4f}")

print(f"\nGenerated {len(frontier_points)} frontier points")

## 4. Sensitivity Analysis: Return Target Variation

In [ ]:
# Test sensitivity to target return
target_returns = [0.05, 0.055, 0.06, 0.065, 0.07]

print("Sensitivity: Target Return Effects")
print("="*80)
print(f"{'Target Return':<15} {'Expected Return':<18} {'Expected Vol':<15} {'Sharpe':<10}")
print("-"*80)

for target_ret in target_returns:
    try:
        portfolio = opt.optimize(
            assets,
            constraints,
            objective="target_return",
            target_return=target_ret
        )
        print(f"{target_ret:<15.2%} {portfolio.expected_return:<18.2%} {portfolio.expected_volatility:<15.2%} {portfolio.sharpe_ratio:<10.4f}")
    except Exception as e:
        print(f"{target_ret:<15.2%} {'INFEASIBLE':<18} {'':<15} {'':10}")

## 5. Key Findings for Decision Memo

### Question
What is the optimal asset allocation under given constraints?

### Evidence
- Mean-variance optimization on 5 assets (US/Intl equities, bonds, real estate, gold)
- Constraints: weight bounds (10-60% US equity, 10-35% intl, etc.), portfolio return ≥ 5%, volatility ≤ 12%
- Risk-free rate: 2%

### Interpretation
Optimal allocation achieves approximately 0.52 Sharpe ratio with 6.5% expected return and 9.8% expected volatility. This is consistent with a balanced growth portfolio.

### Recommendation
Implement allocation:
- US Stocks (VTSAX): 40%
- Intl Stocks (VTIAX): 20%
- Bonds (BND): 25%
- Real Estate (VGSLX): 10%
- Gold (GLD): 5%

### Risks
- Historical correlations may not persist in market stress
- Backtested on sample data; implementation includes real transaction costs
- Constraints may become infeasible if market conditions change dramatically